# T7: Array-Oriented Programming with NumPy
CHE-226, Programming and Data Science

**Learning objective:** create NumPy arrays and read their attributes, replace explicit loops with vectorized arithmetic, broadcasting, and universal functions, and select, reshape, and summarize array data without being caught out by views.

**Teaches:** (KA-01)

*Spans two class meetings; see the class break marker partway through.*

## Agenda: This Class
1. Why arrays? Lists vs. arrays
2. Array attributes
3. Filling arrays, ranges, and reshape
4. Vectorized arithmetic and broadcasting
5. Comparisons and boolean masks

# 1. Why Arrays? Lists vs. Arrays

### Concept
A NumPy array holds numbers of one type in one block of memory. Arithmetic on an array applies to every element at once, with no loop, and runs far faster than a list.

In [ ]:
import numpy as np

T_C = [25.0, 80.0, 150.0]                # a list
T_arr = np.array(T_C)                    # an array from the list
print(T_arr + 273.15)                    # every element, no loop
print([T + 273.15 for T in T_C])         # the list way

grid = np.array([[1, 2, 3], [4, 5, 6]])  # 2D from nested lists
print(grid)

Detailed notes: NumPy is the foundation of Python's scientific stack: pandas, Matplotlib, SciPy, and scikit-learn all build on its arrays. Because every element has the same type and sits in one contiguous block, NumPy runs its loops in compiled C rather than in the Python interpreter; on a million elements this is often 50 to 100 times faster, which students can see live with %timeit sum([x for x in range(10**6)]) versus %timeit np.arange(10**6).sum(). The style is called array-oriented or vectorized programming: describe the operation on the whole array and let NumPy do the looping. np.array takes a list, or a list of lists for a 2D array.

### Activity: predict then run
Same data, same operator. What does each line print?

In [ ]:
lst = [1, 2, 3]
arr = np.array([1, 2, 3])
print(lst * 2)
print(arr * 2)

Answer: [1, 2, 3, 1, 2, 3], then [2 4 6]. For a list, * repeats the sequence; for an array it is element-wise arithmetic. Note the array prints without commas, a quick visual cue for which one you have.

# 2. Array Attributes

### Concept
Every array knows its element type `dtype`, number of dimensions `ndim`, size along each dimension `shape`, and total element count `size`.

In [ ]:
# 3 days (rows) x 4 sensors (columns) of temperatures, K
T = np.array([[350, 352, 351, 355],
              [353, 354, 350, 356],
              [349, 351, 352, 354]])

print(T.dtype, T.ndim)       # element type, number of dimensions
print(T.shape, T.size)       # (rows, columns), total elements
print(T.itemsize)            # bytes per element

Detailed notes: shape is the attribute to check first when anything goes wrong; most NumPy errors are shape mismatches. For a 2D array shape is (rows, columns), so T.shape[0] is the number of days here. dtype is chosen from the data: all integers give int64 (on most systems), and a single float anywhere makes the whole array float64, since an array holds one type. Iterating over a 2D array with for yields rows; T.flat iterates element by element. Attributes are accessed without parentheses because they are stored values, not methods.

### Activity: predict then run
What `dtype` and `shape` does each array have?

In [ ]:
a = np.array([1, 2, 3.5])
b = np.array([[1.0, 2.0]])
print(a.dtype, a.shape)
print(b.dtype, b.shape)

Answer: float64 (3,), then float64 (1, 2). The single 3.5 upcasts all of a to float. b has double brackets, so it is 2D with one row and two columns, not the 1D shape (2,). Students often miss the difference between (2,) and (1, 2).

# 3. Filling Arrays, Ranges, and reshape

### Concept
`zeros`, `ones`, and `full` create filled arrays. `arange` steps by a spacing; `linspace` gives a set number of points. `reshape` changes the shape without changing the data.

In [ ]:
print(np.zeros(3), np.full(3, 298.15))
print(np.arange(300, 350, 10))         # start, stop (excluded), step
print(np.linspace(0.0, 1.0, 5))        # start, stop (included), count

x = np.arange(1, 7)                    # 1..6
print(x.reshape(2, 3))                 # 2 rows x 3 columns

Detailed notes: arange mirrors range: the stop value is excluded and the third argument is the step. With float steps, rounding can make the number of points unpredictable, so linspace, which takes the number of points and includes the end point, is the safer choice for a grid of temperatures or compositions. reshape requires the new shape to have the same number of elements (6 = 2 x 3); -1 in one position means work it out, as in x.reshape(-1, 2). reshape returns a new array object but usually shares the same data (a view, Section 9); resize changes the array in place. Large arrays print with ... in the middle rather than every element.

### Activity: predict then run
Both lines aim for evenly spaced mole fractions from 0 to 1. What does each print?

In [ ]:
print(np.arange(0, 1, 0.25))
print(np.linspace(0, 1, 5))

Answer: [0. 0.25 0.5 0.75], then [0. 0.25 0.5 0.75 1. ]. arange excludes the stop value, so 1.0 is missing; linspace includes it. For a composition sweep that must reach pure component, linspace is right.

# 4. Vectorized Arithmetic and Broadcasting

### Concept
Operators work element by element between two arrays of the same shape. With a single number, NumPy broadcasts it to every element. `+=` and friends update in place.

In [ ]:
m_kg_h = np.array([100.0, 250.0, 80.0])     # stream mass flows
cp = np.array([4.18, 2.0, 1.5])             # kJ/(kg K), per stream
dT = 30.0                                   # same dT for all, K

Q_kW = m_kg_h * cp * dT / 3600              # kJ/h -> kW, all streams
print(Q_kW.round(2))
m_kg_h *= 1.1                               # 10% more flow, in place
print(m_kg_h)

Detailed notes: Element-wise means position i of the result uses position i of each operand, so the arrays must line up. Broadcasting is NumPy's rule for operands of different shapes: a scalar is stretched to match any shape, and more generally dimensions are compared from the right and must be equal or 1. Section 7 shows a 2D case. This replaces the T3 pattern of one formula applied once per list item with a single expression. The heat-transfer sneak peek below uses the same idea: an array of thermal resistances in series is summed in one call.

**Sneak Peek: T-06.04.01 Overall Heat Transfer Coefficient (KA-06)**
Thermal resistances in series add like electrical ones, so an array of resistances gives U in one line. You will study heat exchangers fully in heat transfer.

### Activity: predict then run
Resistances in series add: 1/U = 1/h_i + x/k + 1/h_o. What overall U does this print, roughly?

In [ ]:
h_i, h_o = 500.0, 1500.0          # film coefficients, W/(m^2 K)
x_wall, k_wall = 0.003, 15.0      # wall: thickness m, k W/(m K)

R = np.array([1 / h_i, x_wall / k_wall, 1 / h_o])   # m^2 K / W
U = 1 / R.sum()
print(R.round(6), f"U = {U:.0f} W/(m^2 K)")

Answer: U is about 349 W/(m^2 K). R is [0.002, 0.0002, 0.000667]; the sum is 0.002867, and 1/0.002867 is about 349. The inside film (the largest resistance) dominates, so U ends up below the smallest h. Ask: which coefficient would you try to raise first?

# 5. Comparisons and Boolean Masks

### Concept
A comparison on an array returns an array of True/False, a mask. Indexing with a mask keeps only the True positions. Combine masks with `&` and `|`, in parentheses.

In [ ]:
T = np.array([348.0, 356.0, 351.0, 362.0, 359.0])   # K

hot = T > 355                   # element-wise comparison
print(hot)
print(T[hot])                   # only readings above 355 K
print(T[(T > 350) & (T < 360)]) # both conditions: use &, not and
print(hot.sum())                # True counts as 1

Detailed notes: Masks are the array replacement for an if inside a loop. Use & (and), | (or), and ~ (not) for element-wise logic, always with parentheses around each comparison, because & binds more tightly than >. The plain words and/or raise 'truth value of an array is ambiguous', since Python cannot reduce an array to one True or False; if T > 355 fails the same way, and the fix is (T > 355).any() or .all(). Summing a mask counts the True values, and np.where(mask, a, b) picks element-wise between two options.

### Activity: spot the bug
This should count how many readings exceed the 355 K alarm limit. It runs, but the count is wrong. Why?

In [ ]:
T = np.array([348.0, 356.0, 351.0, 362.0, 359.0])
n_alarms = len(T > 355)
print(f"{n_alarms} readings above the limit")

Answer: prints 5, the length of the mask, which always equals the number of readings. Fix: (T > 355).sum() or np.count_nonzero(T > 355), which give 3.

## Class Break
Covered so far: creating arrays, attributes, ranges and reshape, vectorized arithmetic and broadcasting, boolean masks.

## Agenda: Next Class
6. Calculation methods and the axis argument
7. Universal functions
8. Indexing and slicing 2D arrays
9. Views and copies
10. Putting it together: a sensor array summary

# 6. Calculation Methods and the axis Argument

### Concept
`sum`, `mean`, `min`, `max`, and `std` summarize an array. With `axis=0` they work down the columns; with `axis=1`, across the rows.

In [ ]:
# 3 days (rows) x 4 sensors (columns), K
T = np.array([[350, 352, 351, 355],
              [353, 354, 350, 356],
              [349, 351, 352, 354]])

print(T.mean())               # one number: all 12 readings
print(T.mean(axis=0))         # per sensor (collapse the rows)
print(T.max(axis=1))          # per day (collapse the columns)

Detailed notes: The axis argument names the dimension that gets collapsed. axis=0 runs down the rows, so one result is produced per column (per sensor); axis=1 runs across the columns, giving one result per row (per day). The result's shape is the original shape with that axis removed: (3, 4) becomes (4,) for axis=0. The same argument appears all through pandas and scikit-learn, so it is worth drilling now. std uses the population formula by default (ddof=0); pass ddof=1 for the sample standard deviation.

### Activity: predict then run
Before running, say the shape of each result.

In [ ]:
print(T.sum(axis=0).shape)
print(T.std(axis=1).shape)
print(T.min(axis=0))

Answer: (4,), then (3,), then [349 351 350 354], the lowest reading for each sensor. Reasoning aloud: the named axis disappears from the shape.

# 7. Universal Functions

### Concept
Universal functions (ufuncs) such as `np.sqrt`, `np.exp`, and `np.log` apply element by element and broadcast like operators, including between a row and a column.

In [ ]:
Ea = np.array([40e3, 60e3, 80e3])     # activation energies, J/mol
T = np.array([[300.0], [350.0]])      # a column: shape (2, 1)

k_rel = np.exp(-Ea / (8.314 * T))     # broadcasts to shape (2, 3)
print(k_rel.shape)
print(np.sqrt(np.array([4.0, 9.0])), np.add(1, [2, 3]))

Detailed notes: Ufuncs exist for arithmetic (add, multiply), exponentials and logs (exp, log, log10), trigonometry, comparisons, and more; np.add(a, b) is what a + b calls. The broadcast here is the key new idea: a (2, 1) column against a (3,) row compares shapes from the right (1 vs 3, then 2 vs nothing) and stretches both into a (2, 3) grid, one row per temperature and one column per activation energy. The Arrhenius factor from T4 is reused deliberately. The root-finding sneak peek below uses ufunc arithmetic to run Newton's method on many equations at once.

**Sneak Peek: T-01.04.01 Root Finding (KA-01)**
Newton-Raphson solves f(x) = 0 by repeating one update line. With arrays, one line updates many equations at once. You will study it fully in numerical methods.

### Activity: predict then run
Newton's method, x_new = x - f(x) / f'(x), here solving x^2 = a for three values of a at once. What does it converge to?

In [ ]:
a = np.array([2.0, 9.0, 50.0])     # solve x**2 - a = 0 for each a
x = np.ones(3)                     # initial guesses
for _ in range(6):
    x = x - (x**2 - a) / (2 * x)   # one Newton step, all at once
print(x.round(4))

Answer: [1.4142 3. 7.0711], the square roots of 2, 9, and 50. Each Newton step is one vectorized line acting on all three equations together. Six steps from a guess of 1 is enough here; mention that convergence is not guaranteed for every function or starting guess.

# 8. Indexing and Slicing 2D Arrays

### Concept
`A[row, col]` picks one element. Slices work in each dimension: `A[:, 2]` is a whole column. A list of indices selects specific rows or columns.

In [ ]:
T = np.array([[350, 352, 351, 355],
              [353, 354, 350, 356],
              [349, 351, 352, 354]])     # days x sensors

print(T[1, 2])          # day 2, sensor 3
print(T[0])             # all of day 1
print(T[:, 3])          # sensor 4 across all days
print(T[0:2, [0, 2]])   # days 1-2, sensors 1 and 3

Detailed notes: One pair of brackets with a comma replaces T5's chained flows[1][2]; both work on arrays, but the comma form is the idiom. The column selection T[:, 3], which needed a comprehension with a 2D list, is now one expression: this is the main practical win over lists of lists. Slices use the same start:stop:step rules as lists. Selecting with a list of indices ([0, 2]) is called fancy indexing and always returns a copy, whereas a plain slice returns a view, which is the subject of the next section.

### Activity: predict then run
Using `T` above, what does each line print?

In [ ]:
print(T[2, -1])
print(T[1:, 0])
print(T[:, 1:3].shape)

Answer: 354 (last day, last sensor); [353 349] (days 2 and 3, sensor 1); (3, 2) (all days, sensors 2 and 3).

# 9. Views and Copies

### Concept
A slice of an array is a view: a new window onto the same data. Changing the view changes the original. `.copy()` makes an independent deep copy.

In [ ]:
readings = np.array([350.0, 352.0, 351.0, 355.0])

first_two = readings[:2]        # a view, shares data
first_two[0] = 0.0              # write through the view
print(readings)                 # the original changed

safe = readings[:2].copy()      # an independent copy
safe[1] = -1.0
print(readings)                 # unchanged this time

Detailed notes: This is the key difference from lists, where a slice is a copy. NumPy avoids copying because arrays can be huge; the cost is that edits through a slice leak back into the original. view() and reshape also share data; copy() allocates new memory. np.shares_memory(a, b) or checking b.base tells you which you have. Rule of thumb: if you plan to modify a piece of an array while keeping the original, call .copy() explicitly. The same issue reappears in pandas as the SettingWithCopyWarning.

### Activity: spot the bug
The goal is a separate baseline array in Celsius, leaving `readings` in Kelvin. It runs, but `readings` is wrong afterwards. Why?

In [ ]:
readings = np.array([350.0, 352.0, 351.0, 355.0, 358.0])
baseline = readings[:3]
baseline -= 273.15              # convert the baseline to C
print(readings)

Answer: readings becomes [76.85 78.85 77.85 355. 358.]: baseline is a view, and -= modifies it in place, so the first three Kelvin readings are overwritten. Fix: baseline = readings[:3].copy(), or baseline = readings[:3] - 273.15, which creates a new array.

# 10. Putting It Together: A Sensor Array Summary

### Concept
A flat log of readings becomes a table with `reshape`, is converted with broadcasting, summarized with `axis`, and screened with a mask. `.T` transposes rows and columns.

In [ ]:
log_K = np.array([350, 352, 351, 352, 353, 354, 350, 351,
                  349, 351, 352, 368])        # 12 readings, flat
T = log_K.reshape(3, 4)                       # 3 days x 4 sensors
T_C = T - 273.15                              # broadcast conversion
by_sensor = T_C.T                             # 4 sensors x 3 days
print(by_sensor.mean(axis=1).round(1))        # mean per sensor, C
print(np.argwhere(T > 360))                   # [day, sensor] alarms

Detailed notes: This combines the whole lecture with no loops: reshape turns a flat logger dump into a (days, sensors) table, subtracting 273.15 broadcasts, .T swaps the axes so each row is a sensor, mean(axis=1) summarizes each sensor, and a mask with np.argwhere locates alarms by (row, column). Transposing, like reshape, returns a view. np.hstack and np.vstack are the tools for gluing a new day or a new sensor onto the table. This is also the last stop before pandas in T9, which adds row and column labels to exactly this kind of table.

### Activity: cold call
The mean for sensor 4 is noticeably higher than the others. Is that a hot spot, or one bad reading? How would you check with one line?

In [ ]:
print(T[:, 3], np.median(T[:, 3]))

Answer: one bad reading: sensor 4 reads 352, 351, then 368 K. Its median (352) sits right alongside the other sensors, while the mean is pulled up by the single 368. Strong answers propose looking at the column itself or a median; this echoes the outlier lesson from T4.

## Recap and Lab Practice
- NumPy arrays hold one dtype; arithmetic, comparisons, and ufuncs act on every element with no loop, and broadcasting stretches compatible shapes.
- shape and axis drive everything: axis names the dimension that gets collapsed; A[row, col] and A[:, j] select data.
- Array slices are views, so modify a copy() when the original must survive.

Lab practice: time a list loop against an array operation, vectorize the T3 terminal-velocity loop, and summarize a sensor log per sensor and per day; ask if you want them turned into a lab worksheet.

Next lecture: T8, strings.